In [1]:
# Dependencies
# --------------------------------------------------------------
import os

import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
import statsmodels.api as sm

import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import seaborn as sns

import itertools

import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
# Import Dataset 
# --------------------------------------------------------------

base_path = os.path.join("..", "DATASET/OULAD_ORIGINAL")

studentInfo = pd.read_csv(os.path.join(base_path, "studentInfo.csv"))
assessments = pd.read_csv(os.path.join(base_path, "assessments.csv"))
studentAssessment = pd.read_csv(os.path.join(base_path, "studentAssessment.csv"))
studentVle = pd.read_csv(os.path.join(base_path, "studentVle.csv"))



In [5]:
# Junção das Tabelas
# --------------------------------------------------------------

# Juntar informações das avaliações
dfs = studentInfo.merge(studentAssessment, on="id_student", how="left")

# Juntar com detalhes das avaliações
assessments.drop(columns=["code_module", "code_presentation"], inplace=True) #Remove colunas que vão ser duplicadas
dfs = dfs.merge(assessments, on="id_assessment", how="left")

# Juntar interações com a plataforma
dfs = dfs.merge(studentVle.groupby("id_student")["sum_click"].sum().reset_index(), on="id_student", how="left")
dfs.drop(columns=["id_student", "id_assessment","code_presentation"], inplace=True) #Remover colunas irrelevantes 





studentAssessment

id_assessment      int64
id_student         int64
date_submitted     int64
is_banked          int64
score             object
dtype: object

assessments

id_assessment        int64
assessment_type     object
date                object
weight             float64
dtype: object

studentVle

code_module          object
code_presentation    object
id_student            int64
id_site               int64
date                  int64
sum_click             int64
dtype: object


In [3]:
# Identificação das Features
# --------------------------------------------------------------
# Identificar colunas categóricas
categorical_cols = ["code_module", "gender", "region", "highest_education", "imd_band", "age_band", "disability", "assessment_type", "final_result","is_banked"]
# Selecionar features numericas
numeric_cols =["date_submitted","num_of_prev_attempts", "sum_click","date","studied_credits", "weight","score"]

In [6]:
# Normalização do data set
# --------------------------------------------------------------

# Substituir '?' por NaN
dfs.replace('?', np.nan, inplace=True)

# --------------------------------------------------------------
# numeric_cols
# --------------------------------------------------------------

# Converter colunas numéricas corretamente
dfs[numeric_cols] = dfs[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Preencher valores NaN com a média da respetiva coluna
dfs[numeric_cols] = dfs[numeric_cols].fillna(dfs[numeric_cols].mean())

# --------------------------------------------------------------
# categorical_cols
# --------------------------------------------------------------

# Fazer a moda nas linhas com NaN
for col in categorical_cols:
    mode_value = dfs[col].mode()[0]  # Obtém a moda (valor mais frequente)
    dfs[col] = dfs[col].fillna(mode_value)  # Preenche os NaN com a moda
    
# Remover linhas com 'Withdrawn'
dfs = dfs.loc[dfs['final_result'] != 'Withdrawn']

# Substituir 'Distinction' por 'Pass'
dfs['final_result'] = dfs['final_result'].replace('Distinction', 'Pass')



In [ ]:
# Normalização das Features 
# --------------------------------------------------------------

# Numéricas 
scaler = StandardScaler()
dfs[numeric_cols] = scaler.fit_transform(dfs[numeric_cols])

# Salvar o scaler em um arquivo
joblib.dump(scaler, "scaler.pkl")


# Categóricas
# Criar o dicionário para armazenar os encoders
encoders = {}

# Treinar o encoder para cada coluna e armazená-lo no dicionário
for col in categorical_cols:
    encoder = LabelEncoder()  # Cria um novo LabelEncoder para cada coluna
    encoder.fit(dfs[col])  # Treina o encoder para a coluna
    encoders[col] = encoder  # Armazena o encoder no dicionário

# Salvar os encoders em um arquivo
joblib.dump(encoders, 'encoders.pkl')

In [11]:
# Split feature subsets
# --------------------------------------------------------------

X = dfs.drop(columns="final_result")
y = dfs["final_result"]



In [12]:
# Create a training and test set
# --------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


code_module               int64
gender                    int64
region                    int64
highest_education         int64
imd_band                  int64
age_band                  int64
num_of_prev_attempts    float64
studied_credits         float64
disability                int64
date_submitted          float64
is_banked                 int64
score                   float64
assessment_type           int64
date                    float64
weight                  float64
sum_click               float64
dtype: object
Train set: (127780, 16) (127780,)
Test set: (54764, 16) (54764,)


In [22]:
# Treino do modelo SVM
# --------------------------------------------------------------
svm = SVC() 
svm.fit(X_train, y_train)

SVC()

In [ ]:
# SVC - Hyperparameter tuning (GridSearchCV) - Too slow (+12h)
# --------------------------------------------------------------

# Definição dos hiperparâmetros a testar
param_grid = {
    'C': [0.1, 1, 10, 100],  
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],  
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]  
}

# GridSearchCV para encontrar os melhores hiperparâmetros
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=2)

# Treinar o modelo
grid_search.fit(X_train, y_train)

# Exibir os melhores hiperparâmetros encontrados
print("Melhores Hiperparâmetros:", grid_search.best_params_)
print("Melhor Score:", grid_search.best_score_)

# Modelo otimizado
best_svm = grid_search.best_estimator_

'# SVC - Hyperparameter tuning (GridSearchCV) - too slow\n# --------------------------------------------------------------\n\n# Definição dos hiperparâmetros a testar\nparam_grid = {\n    \'C\': [0.1, 1, 10, 100],  \n    \'kernel\': [\'linear\', \'rbf\', \'poly\', \'sigmoid\'],  \n    \'gamma\': [\'scale\', \'auto\', 0.001, 0.01, 0.1, 1]  \n}\n\n# GridSearchCV para encontrar os melhores hiperparâmetros\ngrid_search = GridSearchCV(svm, param_grid, cv=5, scoring=\'f1_weighted\', n_jobs=-1, verbose=2)\n\n# Treinar o modelo\ngrid_search.fit(X_train, y_train)\n\n# Exibir os melhores hiperparâmetros encontrados\nprint("Melhores Hiperparâmetros:", grid_search.best_params_)\nprint("Melhor Score:", grid_search.best_score_)\n\n# Modelo otimizado\nbest_svm = grid_search.best_estimator_'

In [ ]:
# SVC - Hyperparameter tuning (RandomizedSearchCV) - Too slow (+12h)
# --------------------------------------------------------------
from sklearn.model_selection import RandomizedSearchCV

# Definição dos hiperparâmetros a testar
param_dist = {
    'C': [0.1, 1, 10, 100],  
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],  
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]  
}

# RandomizedSearchCV para encontrar os melhores hiperparâmetros
random_search = RandomizedSearchCV(svm, param_distributions=param_dist, n_iter=50, cv=5, 
                                   scoring='f1_weighted', n_jobs=-1, random_state=42, verbose=3)

# Treinar o modelo
random_search.fit(X_train, y_train)

# Exibir os melhores hiperparâmetros encontrados
print("Melhores Hiperparâmetros:", random_search.best_params_)
print("Melhor Score:", random_search.best_score_)

# Modelo otimizado
best_svm = random_search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END ....................C=100, gamma=scale, kernel=rbf; total time=111.1min
[CV] END ....................C=100, gamma=scale, kernel=rbf; total time=111.8min
[CV] END ....................C=100, gamma=scale, kernel=rbf; total time=112.3min
[CV] END ....................C=100, gamma=scale, kernel=rbf; total time=113.0min
[CV] END ....................C=100, gamma=scale, kernel=rbf; total time=114.4min
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=35.9min
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=36.6min
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=37.0min
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=29.9min
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=40.1min
[CV] END .....................C=100, gamma=auto, kernel=rbf; total time=304.0min
[CV] END ..................C=100, gamma=auto, k

KeyboardInterrupt: 

In [ ]:
# Treino do modelo Random Forest
# --------------------------------------------------------------
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

In [ ]:
# Random Forest - Hyperparameter tuning (GridSearchCV)
# --------------------------------------------------------------

# Definir a grade de hiperparâmetros
param_grid = {
    'n_estimators': [50, 100, 200],   # Número de árvores na floresta
    'max_depth': [None, 10, 20],      # Profundidade máxima da árvore
    'min_samples_split': [2, 5, 10],  # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4]     # Mínimo de amostras em cada folha
}

# Criar o modelo de Random Forest
rf = RandomForestClassifier(random_state=42)

# Aplicar GridSearchCV
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=3)
grid_search.fit(X_train, y_train)

# Melhor combinação de hiperparâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor modelo treinado com os melhores hiperparâmetros
best_rf = grid_search.best_estimator_

# Avaliação no conjunto de teste
y_pred = best_rf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Treino do modelo Rede Neuronal
# --------------------------------------------------------------
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)

In [ ]:
# Rede Neuronal - Hyperparameter tuning (GridSearchCV)
# --------------------------------------------------------------

# Definir a grade de hiperparâmetros
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (150, 100, 50)],  # Diferentes tamanhos de camadas ocultas
    'activation': ['relu', 'tanh', 'logistic'],  # Funções de ativação
    'solver': ['adam', 'sgd'],  # Algoritmos de otimização
    'alpha': [0.0001, 0.001, 0.01],  # Taxa de regularização
    'learning_rate': ['constant', 'adaptive']  # Taxa de aprendizado
}

# Criar o modelo MLP
mlp = MLPClassifier(max_iter=500, random_state=42)

# Aplicar GridSearchCV
grid_search = GridSearchCV(mlp, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=3)
grid_search.fit(X_train, y_train)

# Melhor combinação de hiperparâmetros
print("Melhores parâmetros:", grid_search.best_params_)

# Melhor modelo treinado com os melhores hiperparâmetros
best_mlp = grid_search.best_estimator_

# Avaliação no conjunto de teste
y_pred = best_mlp.predict(X_test)
print(classification_report(y_test, y_pred))

In [17]:
# Avaliador de modelo
# --------------------------------------------------------------
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print("Confusion matrix")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    return f1_score(y_test, y_pred, average='weighted')

In [ ]:
# Avaliação dos modelos internos
# --------------------------------------------------------------

print("\nAvaliação SVM")
svm_f1 = evaluate_model(best_svm, X_test, y_test)

print("\nAvaliação Random Forest")
rf_f1 = evaluate_model(best_rf, X_test, y_test)

print("\nAvaliação Rede Neuronal")
mlp_f1 = evaluate_model(best_mlp, X_test, y_test)


Avaliação Random Forest
Confusion matrix
[[ 7228  2975]
 [  461 44100]]
              precision    recall  f1-score   support

           0       0.94      0.71      0.81     10203
           1       0.94      0.99      0.96     44561

    accuracy                           0.94     54764
   macro avg       0.94      0.85      0.89     54764
weighted avg       0.94      0.94      0.93     54764



In [13]:
# Avaliação dos modelos externos
# --------------------------------------------------------------

base_path = "/Users/amorimriki/Documents/GitHub/Projeto-I-PSA"
path = os.path.join(base_path, "ML_MODEL/ensemble_model_80-20.pkl")
model = joblib.load(path)

print("\nAvaliação do modelo " + os.path.splitext(os.path.basename(path))[0])
model_f1 = evaluate_model(model, X_test, y_test)




Avaliação Random Forest
Confusion matrix
[[ 3760  6443]
 [  483 44078]]
              precision    recall  f1-score   support

           0       0.89      0.37      0.52     10203
           1       0.87      0.99      0.93     44561

    accuracy                           0.87     54764
   macro avg       0.88      0.68      0.72     54764
weighted avg       0.88      0.87      0.85     54764



In [15]:
# Ensemble Learning por Majority Voting
# --------------------------------------------------------------
base_path = "/Users/amorimriki/Documents/GitHub/Projeto-I-PSA"


best_svm = joblib.load(os.path.join(base_path, "ML_MODEL/svm_pipeline.pkl"))
best_rf = joblib.load(os.path.join(base_path, "ML_MODEL/rf_pipeline.pkl"))
best_mlp = joblib.load(os.path.join(base_path, "ML_MODEL/mlp_pipeline.pkl"))


ensemble_model = VotingClassifier(
    estimators=[('svm', best_svm), ('rf', best_rf), ('mlp', best_mlp)], voting='soft'
)
ensemble_model.fit(X_train, y_train)



/Users/amorimriki/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


VotingClassifier(estimators=[('svm',
                              Pipeline(steps=[('preprocessor',
                                               ColumnTransformer(transformers=[('cat',
                                                                                OneHotEncoder(handle_unknown='ignore'),
                                                                                ['code_module',
                                                                                 'gender',
                                                                                 'region',
                                                                                 'highest_education',
                                                                                 'imd_band',
                                                                                 'age_band',
                                                                                 'disability',
                                                                                 'assessment_type',
                                                                                 'is_banked']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['date_submitted',
                                                                                 'num_of_prev_attempts',
                                                                                 'sum_click',
                                                                                 'date',...
                                                                                 'region',
                                                                                 'highest_education',
                                                                                 'imd_band',
                                                                                 'age_band',
                                                                                 'disability',
                                                                                 'assessment_type',
                                                                                 'is_banked']),
                                                                               ('num',
                                                                                StandardScaler(),
                                                                                ['date_submitted',
                                                                                 'num_of_prev_attempts',
                                                                                 'sum_click',
                                                                                 'date',
                                                                                 'studied_credits',
                                                                                 'weight',
                                                                                 'score'])])),
                                              ('classifier',
                                               MLPClassifier(activation='tanh',
                                                             alpha=0.001,
                                                             hidden_layer_sizes=(100,
                                                                                 50),
                                                             max_iter=300,
                                                             random_state=42))]))],
                 voting='soft')

In [16]:
# Export Ensemble Model 
# --------------------------------------------------------------

# Salvar o modelo
joblib.dump(ensemble_model, "../ML_MODEL/ensemble_model_pipeline_tunned.pkl")


['../ML_MODEL/ensemble_model_pipeline_tunned.pkl']

In [18]:
# Avaliação do Ensemble Model 
# --------------------------------------------------------------
print("\nAvaliação Ensemble Model")
ensemble_f1 = evaluate_model(ensemble_model, X_test, y_test)


Avaliação Ensemble Model
Confusion matrix
[[ 7244  2959]
 [  301 44260]]
              precision    recall  f1-score   support

           0       0.96      0.71      0.82     10203
           1       0.94      0.99      0.96     44561

    accuracy                           0.94     54764
   macro avg       0.95      0.85      0.89     54764
weighted avg       0.94      0.94      0.94     54764



In [ ]:
# Ensemble Model Confusion Matrix 
# --------------------------------------------------------------

# Obter previsões do modelo ensemble
y_pred = ensemble_model.predict(X_test)

# Criar a matriz de confusão
conf_matrix = confusion_matrix(y_test, y_pred)

# Criar o heatmap da matriz de confusão
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Greens", linewidths=1, linecolor='black')

# Adicionar rótulos
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão")

# Guardar e exibir o gráfico
plt.savefig("../EDA_IMAGES/confusion_matrix_ensemble_model_pipeline_tunned.png", dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
from sklearn.utils import resample

# Remover a coluna final_result
df_no_target = df.drop(columns=["final_result"])

# Gerar dados sintéticos com base no original (aumentar para 500 amostras, por exemplo)
synthetic_data = resample(df_no_target, replace=True, n_samples=500, random_state=42)

# Guardar o novo dataset num ficheiro CSV
output_path = "/mnt/data/synthetic_student_data.csv"
synthetic_data.to_csv(output_path, index=False)

import pandas as pd
import numpy as np

# Carregar o dataset sintético anterior
df_synthetic = pd.read_csv("/mnt/data/synthetic_student_data.csv")

# Gerar IDs de estudante (pode repetir), com valores entre 100000 e 100099
n_students = 100
student_ids = np.random.choice(range(100000, 100000 + n_students), size=len(df_synthetic), replace=True)

# Adicionar ao dataframe
df_synthetic['n_student'] = student_ids

# Guardar o novo dataset com a coluna de estudante
output_path = "/mnt/data/synthetic_student_data_with_students.csv"
df_synthetic.to_csv(output_path, index=False)

output_path
